# Module 2.2: Episodic Memory

The previous notebook showed that chat history — even with compaction —
is a **record of conversations**, not knowledge. It cannot learn patterns,
resolve contradictions, or personalize across sessions.

This notebook introduces **episodic memory**: the agent decides what's worth
remembering and stores it as structured events. Later, it recalls only the
relevant episodes to personalize its responses.

| Chat History | Episodic Memory |
|---|---|
| Stores every message | Stores only significant events |
| Raw transcript | Structured knowledge (type, details, timestamp) |
| Retrieved by session | Retrieved by relevance |
| Agent replays | Agent reasons |


## Prerequisites

- Everything from Module 2.1 (Azure AI Foundry, Cosmos DB, `.env`)
- The same `travel-memory` Cosmos DB database is reused
- This notebook creates a new container `episodic-events`
  (partitioned by `/user_id`) on first run


In [ ]:
%pip install -q -r ./../requirements.txt

## The Gap (Recap)

Even with `CompactionProvider` + `SummarizationStrategy`, the agent cannot:

- **Learn**: "Sarah always books Marriott" requires extracting a pattern
- **Resolve contradictions**: "Hilton in Jan, Marriott in March" — which is current?
- **Personalize proactively**: recommend based on past experiences

The solution: let the agent **curate** its own memory.


In [ ]:
import sys
import sniffio

sys.path.insert(0, "./..")
sniffio.current_async_library_cvar.set("asyncio")

from shared.travel_agent import (
    create_client, SYSTEM_PROMPT,
    search_flights, search_hotels, get_travel_policy,
)

client, credential = create_client("./../.env")
print("Client ready")

## Step 1: Agent as Curator

Not every message is worth remembering. "Hi" and "Thanks" are noise.
But "I stayed at the Marriott and it was great" is a signal.

We give the agent two tools:
- **`remember_event`** — store a structured event (trip, preference, feedback)
- **`recall_events`** — retrieve past events for a user, filtered by type

The agent decides *when* to call them based on its system prompt.


In [ ]:
import os
from azure.cosmos.aio import CosmosClient
from azure.cosmos import PartitionKey

COSMOS_ENDPOINT = os.environ["COSMOS_ENDPOINT"]

cosmos = CosmosClient(COSMOS_ENDPOINT, credential=credential)
db = await cosmos.create_database_if_not_exists("travel-memory")
episodic_container = await db.create_container_if_not_exists(
    id="episodic-events",
    partition_key=PartitionKey(path="/user_id"),
)
print(f"Cosmos container ready: {COSMOS_ENDPOINT}/travel-memory/episodic-events")

In [ ]:
import json
from uuid import uuid4
from datetime import datetime, timezone
from agent_framework import tool

@tool
async def remember_event(
    user_id: str, event_type: str, description: str, details: str = ""
) -> str:
    """Store a memorable event for a user. event_type: trip|preference|feedback. details: optional JSON string with structured data."""
    parsed_details = {}
    if details and details.strip():
        try:
            parsed_details = json.loads(details)
        except json.JSONDecodeError:
            parsed_details = {"raw": details}
    event = {
        "id": f"{user_id}-{uuid4().hex[:8]}",
        "user_id": user_id,
        "event_type": event_type,
        "description": description,
        "details": parsed_details,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    await episodic_container.upsert_item(event)
    return f"Remembered: {event_type} — {description}"

In [ ]:
@tool
async def recall_events(
    user_id: str, event_type: str = "", limit: int = 5
) -> str:
    """Recall past events for a user. Optionally filter by event_type."""
    query = "SELECT * FROM c WHERE c.user_id = @uid"
    params = [{"name": "@uid", "value": user_id}]
    if event_type:
        query += " AND c.event_type = @etype"
        params.append({"name": "@etype", "value": event_type})
    query += " ORDER BY c.timestamp DESC OFFSET 0 LIMIT @lim"
    params.append({"name": "@lim", "value": limit})

    items = [item async for item in episodic_container.query_items(
        query, parameters=params, partition_key=user_id
    )]
    if not items:
        return f"No events found for {user_id}"
    return json.dumps(items, indent=2, default=str)


### Event Schema

Each event the agent stores has this structure:

```json
{
  "id": "E001-a3f8b2c1",
  "user_id": "E001",
  "event_type": "feedback",
  "description": "Rated Marriott Midtown NYC 5 stars",
  "details": {"hotel": "Marriott Midtown", "city": "New York", "rating": 5},
  "timestamp": "2026-06-29T14:30:00+00:00"
}
```

The agent picks `event_type` and structures `details` — we don't dictate the schema rigidly.


In [ ]:
from agent_framework import Agent

EPISODIC_PROMPT = SYSTEM_PROMPT + "\n\n" + """You also have episodic memory.
WHEN TO REMEMBER:
- User completes or discusses a trip → remember as type "trip"
- User states a preference (airline, hotel, seat) → remember as type "preference"
- User gives feedback or a rating → remember as type "feedback"
WHEN TO RECALL:
- Before recommending hotels or flights → recall past events for the user
- When the user asks about their history → recall relevant events
Always include the user_id (e.g. "E001") when remembering or recalling."""

episodic_agent = Agent(
    client=client,
    name="TravelAssistant",
    instructions=EPISODIC_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy,
           remember_event, recall_events],
)
print("Episodic agent ready (5 tools)")


## Step 2: A Conversation Where the Agent Remembers

Sarah (E001) books a trip to New York. She mentions her hotel experience
and rates it. The agent should autonomously call `remember_event` to store
the key facts.

> **Note**: Since the agent decides what to remember, exact output may vary
> between runs. The important thing is that structured events appear in Cosmos.


In [ ]:
from agent_framework import AgentSession

async def sarah_books_trip():
    session = AgentSession()
    turns = [
        "Hi, I'm Sarah Chen (employee E001). I just got back from New York.",
        "I stayed at the Marriott Midtown — absolutely loved it. 5 out of 5.",
        "For future trips, I always prefer aisle seats and Marriott hotels.",
    ]
    for msg in turns:
        print(f"User: {msg}")
        r = await episodic_agent.run(msg, session=session)
        print(f"Agent: {r.text}\n")

await sarah_books_trip()

In [ ]:
# Inspect what the agent stored in Cosmos
async def inspect_episodes():
    items = [item async for item in episodic_container.query_items(
        "SELECT c.event_type, c.description, c.details, c.timestamp "
        "FROM c WHERE c.user_id = 'E001'",
        partition_key="E001",
    )]
    print(f"Episodes stored: {len(items)}\n")
    for item in items:
        print(json.dumps(item, indent=2, default=str))
        print()

await inspect_episodes()

### What Just Happened

The agent received 3 messages and autonomously decided to store structured
events. Compare:

| Approach | Stored |
|---|---|
| **Chat history** | 6 raw messages (3 user + 3 assistant), unstructured |
| **Episodic memory** | 2–3 curated events with typed fields, queryable |

The agent extracted signal from noise. The greeting ("Hi, I'm Sarah") wasn't
stored as an event — only the actionable facts were.


## Step 3: A New Session — Agent Recalls

A week later, Sarah wants to book another trip. This is a **fresh session** —
no chat history carries over. But the agent has episodic memory.

When Sarah asks for a hotel recommendation, the agent should call
`recall_events` to retrieve her past experiences, then personalize its answer.


In [ ]:
async def sarah_returns():
    session = AgentSession()  # fresh session — no chat history

    print("User: I'm Sarah Chen (E001). I need a hotel in New York again.")
    r = await episodic_agent.run(
        "I'm Sarah Chen (E001). I need a hotel in New York again.",
        session=session,
    )
    print(f"Agent: {r.text}\n")

    print("User: Which hotel would you recommend based on my history?")
    r = await episodic_agent.run(
        "Which hotel would you recommend based on my history?",
        session=session,
    )
    print(f"Agent: {r.text}")

await sarah_returns()

### Key Insight

The agent had **zero chat history** from the first session. Yet it gave
a personalized recommendation because it recalled structured episodes.

This is the fundamental shift:
- Chat history answers "what did we say?"
- Episodic memory answers "what do I know about this user?"


## Comparison: Chat History vs Episodic Memory

| Dimension | Chat History | Episodic Memory |
|---|---|---|
| **What's stored** | Every message | Only significant events |
| **Structure** | Raw text | Typed fields (event_type, details, timestamp) |
| **Retrieval** | By session ID | By user + query + filters |
| **Grows** | Linearly with every turn | Only when something meaningful happens |
| **Learns** | No — just replays | Yes — agent extracts patterns |
| **Cross-session** | Only with same session_id | Always — keyed by user |
| **Cost** | High (all tokens every turn) | Low (only relevant episodes retrieved) |


## What Episodic Memory Still Cannot Do

Episodic memory is powerful for single-user personalization. But it cannot:

- **Cross-user patterns**: "Engineers from Seattle all prefer Marriott"
  → requires aggregating across users
- **Relationship reasoning**: "Sarah's manager approved a higher budget"
  → requires understanding org structure
- **Policy inference**: "VPs get business class; Sarah is now a VP"
  → requires connecting role changes to policy rules

These require a **knowledge graph** — structured relationships between
entities (people, hotels, policies, teams). That's semantic memory.


## Summary

### What We Built

| Component | Purpose |
|---|---|
| `remember_event` tool | Agent stores structured events to Cosmos DB |
| `recall_events` tool | Agent retrieves relevant episodes by user + type |
| Augmented system prompt | Instructs agent *when* to remember and recall |
| `episodic-events` container | Partitioned by `/user_id` for fast queries |

### Architecture So Far

```mermaid
flowchart LR
  U[User] --> A[Agent]
  A --> T[Travel Tools]
  A --> CH[(Chat History<br/>Cosmos: chat-history)]
  A --> EM[(Episodic Memory<br/>Cosmos: episodic-events)]
  CH -.->|raw transcript| A
  EM -->|structured events| A
```

### Next Step

Module 2.3 introduces **semantic memory** — a knowledge graph that captures
relationships between entities. This lets the agent reason across users,
teams, and policies instead of just one person's event history.
